In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("qoute_dataset.csv")


In [8]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [9]:
df.shape

(3038, 2)

In [10]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [11]:
quotes = quotes.str.lower()


In [12]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [13]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer


In [15]:
vocab_size = 10000

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

In [16]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [17]:
sequence = tokinizer.texts_to_sequences(quotes)


In [18]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [19]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [20]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)


In [21]:
len(X)


85271

In [22]:
len(y)


85271

In [23]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [24]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [25]:
y = np.array(y)



In [26]:
X_padded.shape


(85271, 745)

In [27]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [28]:
y.shape


(85271,)

In [29]:
y_one_hot.shape


(85271, 10000)

In [30]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [31]:
embedding_dim = 50
rnn_units = 128

In [32]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [33]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [34]:
rnn_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [35]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [36]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [37]:
lstm_model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [38]:
epochs = 10
batch_size = 128

In [39]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs = epochs,
    batch_size = batch_size,
    validation_split = 0.2

)

Epoch 1/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 44s 74ms/step - accuracy: 0.0381 - loss: 6.8714 - val_accuracy: 0.0483 - val_loss: 6.8034
Epoch 2/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 75s 67ms/step - accuracy: 0.0529 - loss: 6.3855 - val_accuracy: 0.0578 - val_loss: 6.7764
Epoch 3/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 66ms/step - accuracy: 0.0600 - loss: 6.2627 - val_accuracy: 0.0588 - val_loss: 6.7762
Epoch 4/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 66ms/step - accuracy: 0.0709 - loss: 6.0841 - val_accuracy: 0.0703 - val_loss: 6.7188
Epoch 5/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 67ms/step - accuracy: 0.0841 - loss: 5.8874 - val_accuracy: 0.0839 - val_loss: 6.6593
Epoch 6/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 67ms/step - accuracy: 0.0994 - loss: 5.6955 - val_accuracy: 0.0905 - val_loss: 6.6136
Epoch 7/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 67ms/step - accuracy: 0.1093 - loss: 5.5182 - val_accuracy: 0.0923 - val_loss: 6.6061
Epoch 8/10
533/533 ━━━━━━━━━━━━━━━━━━━━ 35s 66ms/step - accuracy: 0.1178 - loss: 5.3493 - 

In [40]:
epochs = 100
batch_size = 128
history_lstm = lstm_model.fit(
    X_padded,
    y_one_hot,
    epochs = epochs,
    batch_size = batch_size,
    validation_split = 0.1

)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 56ms/step - accuracy: 0.0398 - loss: 6.7621 - val_accuracy: 0.0410 - val_loss: 6.6808
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.0621 - loss: 6.3002 - val_accuracy: 0.0725 - val_loss: 6.5084
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.0851 - loss: 6.0255 - val_accuracy: 0.0895 - val_loss: 6.4590
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1005 - loss: 5.8103 - val_accuracy: 0.0979 - val_loss: 6.4186
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.1124 - loss: 5.6173 - val_accuracy: 0.1006 - val_loss: 6.3970
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1221 - loss: 5.4442 - val_accuracy: 0.1073 - val_loss: 6.4166
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1312 - loss: 5.2872 - val_accuracy: 0.1082 - val_loss: 6.4391
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1383 - loss: 5